<a href="https://colab.research.google.com/github/samyak2475/recruiter-copilot/blob/main/AI_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU available: True
GPU name: Tesla T4


In [ ]:
!pip install -q -U pyarrow datasets transformers peft bitsandbytes trl accelerate groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

raw = load_dataset("cnamuangtoun/resume-job-description-fit")
train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()
print(raw)

train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['resume_text', 'job_description_text', 'label'],
        num_rows: 6241
    })
    test: Dataset({
        features: ['resume_text', 'job_description_text', 'label'],
        num_rows: 1759
    })
})


In [ ]:
import re

SKILL_VOCAB = [
    "python", "java", "c++", "c#", "javascript", "typescript", "sql", "r", "scala", "go",
    "html", "css", "react", "node.js", "django", "flask", "spring", "postgresql",
    "mysql", "mongodb", "aws", "azure", "gcp", "docker", "kubernetes", "git", "linux", "rest api",
    "machine learning", "deep learning", "nlp", "computer vision", "pandas", "numpy", "pytorch",
    "tensorflow", "scikit-learn", "power bi", "tableau", "excel", "spark", "hadoop", "etl",
    "data warehousing", "data modeling", "agile", "scrum", "jira", "ci/cd", "devops",
    "project management", "stakeholder management", "financial analysis", "accounting", "gaap",
    "salesforce", "sap", "communication", "leadership", "team building", "public speaking",
    "sales", "customer service", "negotiation", "budgeting", "forecasting", "compliance",
    "risk management", "vendor management", "process improvement", "training", "recruiting",
]

def extract_skills(text):
    text_l = str(text).lower()
    found = set()
    for skill in SKILL_VOCAB:
        pattern = r'(?<![a-z0-9])' + re.escape(skill) + r'(?![a-z0-9])'
        if re.search(pattern, text_l):
            found.add(skill)
    return found

def matched_missing(resume, jd):
    resume_skills = extract_skills(resume)
    jd_skills = extract_skills(jd)
    matched = sorted(resume_skills & jd_skills)
    missing = sorted(jd_skills - resume_skills)
    return matched, missing

In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Key loaded:", os.environ["GROQ_API_KEY"][:5] + "..." if os.environ.get("GROQ_API_KEY") else "NOT FOUND")

Key loaded: gsk_z...


In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
import time

TEACHER_PROMPT = """You are an experienced technical recruiter. Given a candidate's resume \
and a job description, in ONE short sentence (max 30 words) explain WHY this is a "{label}" \
match. Be specific and concrete, referencing actual skills/experience. Do not repeat the label \
itself. Do not use hedging phrases like "it seems" or "overall".

Resume:
{resume}

Job Description:
{jd}

One-sentence verdict:"""

def get_teacher_verdict(resume, jd, label, retries=3):
    prompt = TEACHER_PROMPT.format(label=label, resume=str(resume)[:1500], jd=str(jd)[:1500])
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=80,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f"retry {attempt} after error: {e} (sleeping {wait}s)")
            time.sleep(wait)
    return None

In [ ]:
def sample_stratified(df, n_per_class=500, seed=42):
    return (
        df.groupby("label", group_keys=False)
          .apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=seed))
          .reset_index(drop=True)
    )

distill_df = sample_stratified(train_df, n_per_class=500)
print(len(distill_df), "rows selected")

1500 rows selected


/tmp/ipykernel_443/401959752.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=seed))


In [ ]:
import os
import pandas as pd

CACHE_PATH = "/content/drive/MyDrive/verdict_cache.csv"

if os.path.exists(CACHE_PATH):
    cache_df = pd.read_csv(CACHE_PATH)
    done_idx = set(cache_df["orig_index"])
else:
    cache_df = pd.DataFrame(columns=["orig_index", "verdict"])
    done_idx = set()

print(f"Already done: {len(done_idx)} rows — resuming from there")

rows = []
for i, row in distill_df.iterrows():
    if i in done_idx:
        continue
    verdict = get_teacher_verdict(row["resume_text"], row["job_description_text"], row["label"])
    if verdict:
        rows.append({"orig_index": i, "verdict": verdict})
    if len(rows) % 50 == 0 and len(rows) > 0:
        cache_df = pd.concat([cache_df, pd.DataFrame(rows)], ignore_index=True)
        cache_df.to_csv(CACHE_PATH, index=False)
        rows = []
        print(f"checkpointed at {len(cache_df)} rows")
    time.sleep(3.5)

if rows:
    cache_df = pd.concat([cache_df, pd.DataFrame(rows)], ignore_index=True)
    cache_df.to_csv(CACHE_PATH, index=False)

print("Total distilled verdicts:", len(cache_df))

Already done: 1500 rows — resuming from there


KeyboardInterrupt: 

In [ ]:
try:
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": "test"}],
        max_tokens=10,
    )
    print("SUCCESS:", resp.choices[0].message.content)
except Exception as e:
    print("FULL ERROR:", e)

SUCCESS: Your input is brief but useful for me to respond


In [ ]:
import time

try:
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": "test"}],
        max_tokens=10,
    )
    print("SUCCESS:", resp.choices[0].message.content)
except Exception as e:
    print("ERROR TYPE:", type(e))
    print("FULL ERROR:", str(e))
    if hasattr(e, "response") and e.response is not None:
        print("HEADERS:", dict(e.response.headers))

SUCCESS: It appears to be a simple test. If you


In [ ]:
import pandas as pd

cache_df = pd.read_csv("/content/drive/MyDrive/verdict_cache.csv")
print(len(cache_df), "total verdicts")
cache_df.sample(5)

1500 total verdicts


,orig_index,verdict
129,5079,This candidate is a strong fit due to their ex...
1277,4191,This candidate is a potential fit due to their...
809,25,The candidate lacks relevant accounting experi...
1053,4357,This candidate is a strong fit due to their 6+...
687,1459,The candidate lacks relevant accounting experi...


In [ ]:
import json

def build_target_json(orig_index, label, resume, jd, verdict):
    matched, missing = matched_missing(resume, jd)
    obj = {
        "fit": label,
        "matched_skills": matched[:8],
        "missing_skills": missing[:8],
        "verdict": verdict,
    }
    return json.dumps(obj, ensure_ascii=False)

# quick test on one row
row = train_df.iloc[594]
verdict_row = cache_df[cache_df["orig_index"] == 594].iloc[0]["verdict"]

test_target = build_target_json(594, row["label"], row["resume_text"], row["job_description_text"], verdict_row)
print(test_target)

{"fit": "No Fit", "matched_skills": ["excel"], "missing_skills": ["accounting", "sales"], "verdict": "The candidate lacks direct experience in construction accounting, and their background in processing assessor and tax roll data, as well as programming in a mainframe environment, does not align with the required skills for the Construction Accountant role."}


In [ ]:
SYSTEM_PROMPT = (
    "You are an AI recruiting assistant. Given a resume and a job description, respond with "
    "ONLY a JSON object with keys: fit (\"Good Fit\", \"Potential Fit\", or \"No Fit\"), "
    "matched_skills (list), missing_skills (list), verdict (one sentence). No extra text."
)

def build_user_prompt(resume, jd):
    return f"Resume:\n{str(resume)[:1500]}\n\nJob Description:\n{str(jd)[:1500]}"

training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
            {"role": "assistant", "content": target},
        ]
    })

print(len(training_rows), "training examples built")
print()
print(training_rows[0]["messages"][1]["content"][:300])
print("...")
print(training_rows[0]["messages"][2]["content"])

1500 training examples built

Resume:
ProfileSecure a position with a well-established organization with a stable environment that will lead to a lasting in the field of Electrical Engineering.I want to gain experience and to be a part of a dynamic team. I want to apply all that I have learned through experience and academics in
...
{"fit": "Good Fit", "matched_skills": [], "missing_skills": [], "verdict": "The candidate's experience in operation and maintenance of electrical systems, report analysis, and troubleshooting aligns with the job's requirements for performing routine tests, troubleshooting manufacturing problems, and executing tests according to procedures."}


In [ ]:
# check for duplicate orig_index entries — a likely cause of misaligned verdicts
dupes = cache_df[cache_df.duplicated("orig_index", keep=False)]
print("Duplicate orig_index rows:", len(dupes))
print(dupes.sort_values("orig_index").head(10))

Duplicate orig_index rows: 0
Empty DataFrame
Columns: [orig_index, verdict]
Index: []


In [ ]:
mismatch_row = cache_df.iloc[0]
idx = int(mismatch_row["orig_index"])

print("orig_index:", idx)
print()
print("--- Resume (first 300 chars) ---")
print(train_df.iloc[idx]["resume_text"][:300])
print()
print("--- Label ---")
print(train_df.iloc[idx]["label"])
print()
print("--- Verdict from cache ---")
print(mismatch_row["verdict"])

orig_index: 6191

--- Resume (first 300 chars) ---
ProfileSecure a position with a well-established organization with a stable environment that will lead to a lasting in the field of Electrical Engineering.I want to gain experience and to be a part of a dynamic team. I want to apply all that I have learned through experience and academics in enginee

--- Label ---
Good Fit

--- Verdict from cache ---
The candidate's experience in operation and maintenance of electrical systems, report analysis, and troubleshooting aligns with the job's requirements for performing routine tests, troubleshooting manufacturing problems, and executing tests according to procedures.


In [ ]:
.groupby("label", group_keys=False).apply(lambda g: g.sample(...)).reset_index(drop=True)

SyntaxError: invalid syntax (2760570397.py, line 1)

In [ ]:
def sample_stratified(df, n_per_class=500, seed=42):
    sampled = (
        df.groupby("label", group_keys=False)
          .apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=seed))
    )
    return sampled  # note: NOT reset_index — we keep train_df's real original index

distill_df = sample_stratified(train_df, n_per_class=500)
print(len(distill_df), "rows selected")
print(distill_df.index[:10])

1500 rows selected
Index([6191, 5855, 5952, 5260, 5797, 5281, 6210, 6021, 5901, 6019], dtype='int64')


/tmp/ipykernel_443/110469139.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=seed))


In [ ]:
# repair the existing cache: old orig_index values were positions (0..1499) in
# distill_df's original *order*, not real train_df row numbers
position_to_real_index = {pos: real_idx for pos, real_idx in enumerate(distill_df.index)}

cache_df["orig_index"] = cache_df["orig_index"].map(position_to_real_index)

print(cache_df["orig_index"].isna().sum(), "rows failed to map (should be 0)")
cache_df.head(3)

1234 rows failed to map (should be 0)


,orig_index,verdict
0,NaN,The candidate's experience in operation and ma...
1,NaN,This candidate is a strong fit due to their ex...
2,NaN,The candidate's experience with Business Objec...


In [ ]:
fixed_row = cache_df.iloc[0]
idx = int(fixed_row["orig_index"])

print("orig_index:", idx)
print("--- Resume (first 300 chars) ---")
print(train_df.iloc[idx]["resume_text"][:300])
print("--- Verdict ---")
print(fixed_row["verdict"])

ValueError: cannot convert float NaN to integer

In [ ]:
CACHE_PATH = "/content/drive/MyDrive/verdict_cache.csv"
cache_df.to_csv(CACHE_PATH, index=False)
print("Corrected cache saved.")

Corrected cache saved.


In [ ]:
training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
            {"role": "assistant", "content": target},
        ]
    })

print(len(training_rows), "training examples built")
print()
print(training_rows[0]["messages"][1]["content"][:300])
print("...")
print(training_rows[0]["messages"][2]["content"])

ValueError: cannot convert float NaN to integer

In [ ]:
empty_both = sum(
    1 for r in training_rows
    if json.loads(r["messages"][2]["content"])["matched_skills"] == []
    and json.loads(r["messages"][2]["content"])["missing_skills"] == []
)
print(f"{empty_both} / {len(training_rows)} examples have both skill lists empty")

0 / 0 examples have both skill lists empty


In [ ]:
from datasets import Dataset

sft_dataset = Dataset.from_list(training_rows)
sft_dataset = sft_dataset.train_test_split(test_size=0.1, seed=42)
print(sft_dataset)

DatasetDict({
    train: Dataset({
        features: [],
        num_rows: 0
    })
    test: Dataset({
        features: [],
        num_rows: 0
    })
})


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    bf16=True,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"],
    formatting_func=formatting_func,
)

trainer.train()

StopIteration: 

In [ ]:
print(len(sft_dataset["train"]))
print(len(sft_dataset["test"]))

0
0


new


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')

GPU available: True
Mounted at /content/drive


In [ ]:
from datasets import load_dataset
raw = load_dataset("cnamuangtoun/resume-job-description-fit")
train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()

train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:
import re

SKILL_VOCAB = [
    "python", "java", "c++", "c#", "javascript", "typescript", "sql", "r", "scala", "go",
    "html", "css", "react", "node.js", "django", "flask", "spring", "postgresql",
    "mysql", "mongodb", "aws", "azure", "gcp", "docker", "kubernetes", "git", "linux", "rest api",
    "machine learning", "deep learning", "nlp", "computer vision", "pandas", "numpy", "pytorch",
    "tensorflow", "scikit-learn", "power bi", "tableau", "excel", "spark", "hadoop", "etl",
    "data warehousing", "data modeling", "agile", "scrum", "jira", "ci/cd", "devops",
    "project management", "stakeholder management", "financial analysis", "accounting", "gaap",
    "salesforce", "sap", "communication", "leadership", "team building", "public speaking",
    "sales", "customer service", "negotiation", "budgeting", "forecasting", "compliance",
    "risk management", "vendor management", "process improvement", "training", "recruiting",
]

def extract_skills(text):
    text_l = str(text).lower()
    found = set()
    for skill in SKILL_VOCAB:
        pattern = r'(?<![a-z0-9])' + re.escape(skill) + r'(?![a-z0-9])'
        if re.search(pattern, text_l):
            found.add(skill)
    return found

def matched_missing(resume, jd):
    resume_skills = extract_skills(resume)
    jd_skills = extract_skills(jd)
    matched = sorted(resume_skills & jd_skills)
    missing = sorted(jd_skills - resume_skills)
    return matched, missing

def build_target_json(orig_index, label, resume, jd, verdict):
    import json
    matched, missing = matched_missing(resume, jd)
    obj = {"fit": label, "matched_skills": matched[:8], "missing_skills": missing[:8], "verdict": verdict}
    return json.dumps(obj, ensure_ascii=False)

In [ ]:
import pandas as pd
cache_df = pd.read_csv("/content/drive/MyDrive/verdict_cache.csv")
print(len(cache_df), "verdicts loaded from Drive")

1500 verdicts loaded from Drive


In [ ]:
SYSTEM_PROMPT = (
    "You are an AI recruiting assistant. Given a resume and a job description, respond with "
    "ONLY a JSON object with keys: fit (\"Good Fit\", \"Potential Fit\", or \"No Fit\"), "
    "matched_skills (list), missing_skills (list), verdict (one sentence). No extra text."
)

def build_user_prompt(resume, jd):
    return f"Resume:\n{str(resume)[:1500]}\n\nJob Description:\n{str(jd)[:1500]}"

training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
            {"role": "assistant", "content": target},
        ]
    })

from datasets import Dataset
sft_dataset = Dataset.from_list(training_rows)
sft_dataset = sft_dataset.train_test_split(test_size=0.1, seed=42)
print(sft_dataset)

ValueError: cannot convert float NaN to integer

In [ ]:
print("Total rows:", len(cache_df))
print("Rows with NaN orig_index:", cache_df["orig_index"].isna().sum())
cache_df[cache_df["orig_index"].isna()]

Total rows: 1500
Rows with NaN orig_index: 1234


,orig_index,verdict
0,NaN,The candidate's experience in operation and ma...
1,NaN,This candidate is a strong fit due to their ex...
2,NaN,The candidate's experience with Business Objec...
3,NaN,The candidate's 10+ years of experience in har...
4,NaN,This candidate is a strong fit due to their 5+...
...,...,...
1495,NaN,This candidate is a potential fit due to their...
1496,NaN,This candidate is a strong fit due to their ex...
1497,NaN,This candidate is a strong fit due to their 4+...
1498,NaN,This candidate is a Potential Fit due to their...


In [ ]:
def sample_stratified(df, n_per_class=500, seed=42):
    return (
        df.groupby("label", group_keys=False)
          .apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=seed))
    )

distill_df = sample_stratified(train_df, n_per_class=500)
print(len(distill_df), "rows")

1500 rows


/tmp/ipykernel_842/365664149.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=seed))


In [ ]:
cache_df["orig_index"] = distill_df.index[:len(cache_df)]
print(cache_df["orig_index"].isna().sum(), "NaN remaining (should be 0)")
cache_df.head(3)

0 NaN remaining (should be 0)


,orig_index,verdict
0,6191,The candidate's experience in operation and ma...
1,5855,This candidate is a strong fit due to their ex...
2,5952,The candidate's experience with Business Objec...


In [ ]:
check_row = cache_df.iloc[0]
idx = int(check_row["orig_index"])
print("--- Resume ---")
print(train_df.iloc[idx]["resume_text"][:300])
print("--- Verdict ---")
print(check_row["verdict"])

--- Resume ---
ProfileSecure a position with a well-established organization with a stable environment that will lead to a lasting in the field of Electrical Engineering.I want to gain experience and to be a part of a dynamic team. I want to apply all that I have learned through experience and academics in enginee
--- Verdict ---
The candidate's experience in operation and maintenance of electrical systems, report analysis, and troubleshooting aligns with the job's requirements for performing routine tests, troubleshooting manufacturing problems, and executing tests according to procedures.


In [ ]:
CACHE_PATH = "/content/drive/MyDrive/verdict_cache_v2.csv"
cache_df.to_csv(CACHE_PATH, index=False)
print("Saved as v2.")

Saved as v2.


In [ ]:
training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
            {"role": "assistant", "content": target},
        ]
    })

from datasets import Dataset
sft_dataset = Dataset.from_list(training_rows)
sft_dataset = sft_dataset.train_test_split(test_size=0.1, seed=42)
print(sft_dataset)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 150
    })
})


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    bf16=True,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"],
    formatting_func=formatting_func,
)

trainer.train()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ckpt_dir = "/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora"
if os.path.exists(ckpt_dir):
    print(os.listdir(ckpt_dir))
else:
    print("Output directory doesn't exist yet")

Mounted at /content/drive
['README.md', 'checkpoint-85']


In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


new

Cell 1: !nvidia-smi (quick sanity check GPU's still there)

Cell: !pip install -q -U bitsandbytes peft trl accelerate

Cell 2: mount Drive

Cell 3: load dataset

Cell 4: skill vocab + functions

Cell 5: load cache + rebuild training set

Cell 6: reload base model in 4-bit

Cell 7: attach LoRA

Cell 8: resume training

In [ ]:
!nvidia-smi

Mon Jul 27 17:47:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U bitsandbytes peft trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
raw = load_dataset("cnamuangtoun/resume-job-description-fit")
train_df = raw["train"].to_pandas()

train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:
import re, json

SKILL_VOCAB = [
    "python", "java", "c++", "c#", "javascript", "typescript", "sql", "r", "scala", "go",
    "html", "css", "react", "node.js", "django", "flask", "spring", "postgresql",
    "mysql", "mongodb", "aws", "azure", "gcp", "docker", "kubernetes", "git", "linux", "rest api",
    "machine learning", "deep learning", "nlp", "computer vision", "pandas", "numpy", "pytorch",
    "tensorflow", "scikit-learn", "power bi", "tableau", "excel", "spark", "hadoop", "etl",
    "data warehousing", "data modeling", "agile", "scrum", "jira", "ci/cd", "devops",
    "project management", "stakeholder management", "financial analysis", "accounting", "gaap",
    "salesforce", "sap", "communication", "leadership", "team building", "public speaking",
    "sales", "customer service", "negotiation", "budgeting", "forecasting", "compliance",
    "risk management", "vendor management", "process improvement", "training", "recruiting",
]

def extract_skills(text):
    text_l = str(text).lower()
    found = set()
    for skill in SKILL_VOCAB:
        pattern = r'(?<![a-z0-9])' + re.escape(skill) + r'(?![a-z0-9])'
        if re.search(pattern, text_l):
            found.add(skill)
    return found

def matched_missing(resume, jd):
    resume_skills = extract_skills(resume)
    jd_skills = extract_skills(jd)
    return sorted(resume_skills & jd_skills), sorted(jd_skills - resume_skills)

def build_target_json(idx, label, resume, jd, verdict):
    matched, missing = matched_missing(resume, jd)
    return json.dumps({"fit": label, "matched_skills": matched[:8], "missing_skills": missing[:8], "verdict": verdict}, ensure_ascii=False)

In [ ]:
import pandas as pd
from datasets import Dataset

cache_df = pd.read_csv("/content/drive/MyDrive/verdict_cache_v2.csv")

SYSTEM_PROMPT = (
    "You are an AI recruiting assistant. Given a resume and a job description, respond with "
    "ONLY a JSON object with keys: fit (\"Good Fit\", \"Potential Fit\", or \"No Fit\"), "
    "matched_skills (list), missing_skills (list), verdict (one sentence). No extra text."
)

def build_user_prompt(resume, jd):
    return f"Resume:\n{str(resume)[:1500]}\n\nJob Description:\n{str(jd)[:1500]}"

training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
        {"role": "assistant", "content": target},
    ]})

sft_dataset = Dataset.from_list(training_rows).train_test_split(test_size=0.1, seed=42)
print(sft_dataset)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 150
    })
})


In [ ]:
import bitsandbytes
print(bitsandbytes.__version__)

0.50.0


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora",
    per_device_train_batch_size=4, gradient_accumulation_steps=4, num_train_epochs=3,
    learning_rate=2e-4, logging_steps=10, eval_strategy="steps", eval_steps=50,
    save_strategy="epoch", bf16=True, max_length=1024, report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"], formatting_func=formatting_func)

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora/checkpoint-85")

Applying formatting function to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.676453,1.683976,1.755116,168592.000000,0.642411


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.676453,1.683976,1.755116,168592.000000,0.642411
150,1.379819,1.332004,1.330635,727065.000000,0.715885


In [ ]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    fp16=True,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"], formatting_func=formatting_func)

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora/checkpoint-170")

Applying formatting function to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

ValueError: You cannot perform fine-tuning on purely quantized models. Please attach trainable adapters on top of the quantized model to correctly perform fine-tuning. Please see: https://huggingface.co/docs/transformers/peft for more details

final

In [ ]:
# 1. Install everything needed
!pip install -q -U pyarrow datasets transformers peft bitsandbytes trl accelerate

import torch, os, re, json
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# 2. Load dataset
from datasets import load_dataset, Dataset
raw = load_dataset("cnamuangtoun/resume-job-description-fit")
train_df = raw["train"].to_pandas()

# 3. Skill vocab + helpers
SKILL_VOCAB = [
    "python", "java", "c++", "c#", "javascript", "typescript", "sql", "r", "scala", "go",
    "html", "css", "react", "node.js", "django", "flask", "spring", "postgresql",
    "mysql", "mongodb", "aws", "azure", "gcp", "docker", "kubernetes", "git", "linux", "rest api",
    "machine learning", "deep learning", "nlp", "computer vision", "pandas", "numpy", "pytorch",
    "tensorflow", "scikit-learn", "power bi", "tableau", "excel", "spark", "hadoop", "etl",
    "data warehousing", "data modeling", "agile", "scrum", "jira", "ci/cd", "devops",
    "project management", "stakeholder management", "financial analysis", "accounting", "gaap",
    "salesforce", "sap", "communication", "leadership", "team building", "public speaking",
    "sales", "customer service", "negotiation", "budgeting", "forecasting", "compliance",
    "risk management", "vendor management", "process improvement", "training", "recruiting",
]

def extract_skills(text):
    text_l = str(text).lower()
    found = set()
    for skill in SKILL_VOCAB:
        pattern = r'(?<![a-z0-9])' + re.escape(skill) + r'(?![a-z0-9])'
        if re.search(pattern, text_l):
            found.add(skill)
    return found

def matched_missing(resume, jd):
    resume_skills = extract_skills(resume)
    jd_skills = extract_skills(jd)
    return sorted(resume_skills & jd_skills), sorted(jd_skills - resume_skills)

def build_target_json(idx, label, resume, jd, verdict):
    matched, missing = matched_missing(resume, jd)
    return json.dumps({"fit": label, "matched_skills": matched[:8], "missing_skills": missing[:8], "verdict": verdict}, ensure_ascii=False)

# 4. Load verdict cache + rebuild training set
cache_df = pd.read_csv("/content/drive/MyDrive/verdict_cache_v2.csv")

SYSTEM_PROMPT = (
    "You are an AI recruiting assistant. Given a resume and a job description, respond with "
    "ONLY a JSON object with keys: fit (\"Good Fit\", \"Potential Fit\", or \"No Fit\"), "
    "matched_skills (list), missing_skills (list), verdict (one sentence). No extra text."
)

def build_user_prompt(resume, jd):
    return f"Resume:\n{str(resume)[:1500]}\n\nJob Description:\n{str(jd)[:1500]}"

training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
        {"role": "assistant", "content": target},
    ]})

sft_dataset = Dataset.from_list(training_rows).train_test_split(test_size=0.1, seed=42)
print("Dataset ready:", sft_dataset)

# 5. Load base model in 4-bit (fp16, since T4 doesn't support bf16)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

# 6. Attach LoRA
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("EVERYTHING READY — proceed to training cell next")

Mounted at /content/drive


train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [ ]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    fp16=True,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"], formatting_func=formatting_func)

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora/checkpoint-170")

Applying formatting function to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
[transformers] Warning: The following arguments do not match the ones in the `trainer_state.json` within the checkpoint directory: 
	eval_steps: 100 (from args) != 50 (from trainer_state.json)
	save_steps: 25 (from args) != 500 (from trainer_state.json)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-final")
print("Saved final model to Drive.")

Saved final model to Drive.


In [ ]:
def generate(model, tokenizer, resume, jd, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(resume, jd)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text

In [ ]:
test_df = raw["test"].to_pandas()
sample = test_df.iloc[0]

print("=== TRUE LABEL ===")
print(sample["label"])
print()
print("=== FINE-TUNED MODEL OUTPUT ===")
print(generate(model, tokenizer, sample["resume_text"], sample["job_description_text"]))

=== TRUE LABEL ===
No Fit

=== FINE-TUNED MODEL OUTPUT ===
{"fit": "Good Fit", "matched_skills": ["communication"], "missing_skills": [], "verdict": "This candidate is a strong fit due to their extensive experience in data warehousing, ETL design, and BI reporting, which aligns with the job's requirement for technical expertise and problem-solving skills."}


In [ ]:
import re as _re

def try_parse_json(text):
    match = _re.search(r"\{.*\}", text, _re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

eval_sample = test_df.sample(30, random_state=0).reset_index(drop=True)

correct = 0
valid_json = 0
results = []

for i, row in eval_sample.iterrows():
    raw_out = generate(model, tokenizer, row["resume_text"], row["job_description_text"])
    parsed = try_parse_json(raw_out)
    predicted = parsed["fit"] if parsed and "fit" in parsed else "PARSE_ERROR"
    if parsed:
        valid_json += 1
    if predicted == row["label"]:
        correct += 1
    results.append({"true": row["label"], "predicted": predicted})
    print(f"{i+1}/30 — true: {row['label']:15s} predicted: {predicted}")

print()
print(f"JSON-validity rate: {valid_json}/30 = {valid_json/30:.1%}")
print(f"Label accuracy: {correct}/30 = {correct/30:.1%}")

1/30 — true: Good Fit        predicted: No Fit
2/30 — true: No Fit          predicted: Good Fit
3/30 — true: No Fit          predicted: Good Fit
4/30 — true: No Fit          predicted: Good Fit
5/30 — true: Good Fit        predicted: Good Fit
6/30 — true: Good Fit        predicted: Good Fit
7/30 — true: Good Fit        predicted: Good Fit
8/30 — true: Potential Fit   predicted: Good Fit
9/30 — true: Good Fit        predicted: No Fit
10/30 — true: Good Fit        predicted: Good Fit
11/30 — true: No Fit          predicted: Good Fit
12/30 — true: No Fit          predicted: Good Fit
13/30 — true: Good Fit        predicted: Good Fit
14/30 — true: No Fit          predicted: No Fit
15/30 — true: No Fit          predicted: Good Fit
16/30 — true: No Fit          predicted: Good Fit
17/30 — true: No Fit          predicted: Good Fit
18/30 — true: No Fit          predicted: Good Fit
19/30 — true: No Fit          predicted: No Fit
20/30 — true: No Fit          predicted: Good Fit
21/30 — true: Pot

In [ ]:
# check the actual label distribution in what we trained on
trained_labels = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    trained_labels.append(train_df.iloc[idx]["label"])

import pandas as pd
print(pd.Series(trained_labels).value_counts())

Good Fit         500
No Fit           500
Potential Fit    500
Name: count, dtype: int64


In [ ]:
# grab a few actual training examples (ones the model has seen)
train_check = cache_df.sample(10, random_state=1)

correct_train = 0
for _, vrow in train_check.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    raw_out = generate(model, tokenizer, row["resume_text"], row["job_description_text"])
    parsed = try_parse_json(raw_out)
    predicted = parsed["fit"] if parsed else "PARSE_ERROR"
    match = "✓" if predicted == row["label"] else "✗"
    print(f"{match} true: {row['label']:15s} predicted: {predicted}")
    if predicted == row["label"]:
        correct_train += 1

print(f"\nTraining-set accuracy: {correct_train}/10")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


KeyboardInterrupt: 

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,   # force fp16 everywhere, T4 doesn't support bf16
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [ ]:
# force LoRA adapter weights to fp16, since T4 can't handle bfloat16
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

print("LoRA params cast to fp16.")

LoRA params cast to fp16.


In [ ]:
sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v2",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"], formatting_func=formatting_func)

  trainer.train()

Applying formatting function to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.487409,1.521801,1.604504,1111231.000000,0.683821
200,0.741748,0.886082,0.948251,2219085.000000,0.818967
300,0.540789,0.633052,0.648249,3329434.000000,0.873206


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.487409,1.521801,1.604504,1111231.000000,0.683821
200,0.741748,0.886082,0.948251,2219085.000000,0.818967
300,0.540789,0.633052,0.648249,3329434.000000,0.873206
400,0.390074,0.527209,0.516578,4441101.000000,0.896917


GitHub

In [ ]:
!git config --global user.name "samyak2475"
!git config --global user.email "hadgekarsamyak04@gmail.com"

In [ ]:
%cd /content

!git init

/content
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [ ]:
!git branch -M main

In [ ]:
!git remote add origin https://github.com/samyak2475/recruiter-copilot.git

In [ ]:
!git remote remove origin
!git remote add origin https://github.com/samyak2475/recruiter-copilot.git

In [ ]:
!git add .

In [ ]:
!git commit -m "Initial commit"

[main (root-commit) 5bc75e9] Initial commit
 21 files changed, 51059 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.06.04/13.38.54.010366.log
 create mode 100644 .config/logs/2026.06.04/13.39.12.467650.log
 create mode 100644 .config/logs/2026.06.04/13.39.22.644527.log
 create mode 100644 .config/logs/2026.06.04/13.39.24.274402.log
 create mode 100644 .config/logs/2026.06.04/13.39.36.079406.log
 create mode 100644 .config/logs/2026.06.04/13.39.37.138439.log
 create mode 100755 sample_data/README.md
 create mode 100755

In [ ]:
!git push -u origin main

fatal: could not read Username for 'https://github.com': No such device or address


Final 2

In [1]:
!pip install -q -U pyarrow datasets transformers peft bitsandbytes trl accelerate

import torch, os, re, json
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset, Dataset
raw = load_dataset("cnamuangtoun/resume-job-description-fit")
train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()

SKILL_VOCAB = [
    "python", "java", "c++", "c#", "javascript", "typescript", "sql", "r", "scala", "go",
    "html", "css", "react", "node.js", "django", "flask", "spring", "postgresql",
    "mysql", "mongodb", "aws", "azure", "gcp", "docker", "kubernetes", "git", "linux", "rest api",
    "machine learning", "deep learning", "nlp", "computer vision", "pandas", "numpy", "pytorch",
    "tensorflow", "scikit-learn", "power bi", "tableau", "excel", "spark", "hadoop", "etl",
    "data warehousing", "data modeling", "agile", "scrum", "jira", "ci/cd", "devops",
    "project management", "stakeholder management", "financial analysis", "accounting", "gaap",
    "salesforce", "sap", "communication", "leadership", "team building", "public speaking",
    "sales", "customer service", "negotiation", "budgeting", "forecasting", "compliance",
    "risk management", "vendor management", "process improvement", "training", "recruiting",
]

def extract_skills(text):
    text_l = str(text).lower()
    found = set()
    for skill in SKILL_VOCAB:
        pattern = r'(?<![a-z0-9])' + re.escape(skill) + r'(?![a-z0-9])'
        if re.search(pattern, text_l):
            found.add(skill)
    return found

def matched_missing(resume, jd):
    resume_skills = extract_skills(resume)
    jd_skills = extract_skills(jd)
    return sorted(resume_skills & jd_skills), sorted(jd_skills - resume_skills)

def build_target_json(idx, label, resume, jd, verdict):
    matched, missing = matched_missing(resume, jd)
    return json.dumps({"fit": label, "matched_skills": matched[:8], "missing_skills": missing[:8], "verdict": verdict}, ensure_ascii=False)

cache_df = pd.read_csv("/content/drive/MyDrive/verdict_cache_v2.csv")

SYSTEM_PROMPT = (
    "You are an AI recruiting assistant. Given a resume and a job description, respond with "
    "ONLY a JSON object with keys: fit (\"Good Fit\", \"Potential Fit\", or \"No Fit\"), "
    "matched_skills (list), missing_skills (list), verdict (one sentence). No extra text."
)

def build_user_prompt(resume, jd):
    return f"Resume:\n{str(resume)[:1500]}\n\nJob Description:\n{str(jd)[:1500]}"

training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
        {"role": "assistant", "content": target},
    ]})

sft_dataset = Dataset.from_list(training_rows).train_test_split(test_size=0.1, seed=42)
print("Dataset ready:", sft_dataset)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()

def generate(model, tokenizer, resume, jd, max_new_tokens=200):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": build_user_prompt(resume, jd)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def try_parse_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

ckpt_dir = "/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v2"
if os.path.exists(ckpt_dir):
    print("Existing v2 checkpoints:", sorted(os.listdir(ckpt_dir)))
else:
    print("No v2 checkpoints yet — starting fresh")

print("EVERYTHING READY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 18.2 MB/s eta 0:00:00
Mounted at /content/drive


train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364
Existing v2 checkpoints: ['README.md', 'checkpoint-500', 'checkpoint-510']
EVERYTHING READY


In [ ]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v2",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"], formatting_func=formatting_func)

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v2/checkpoint-450")

Applying formatting function to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,0.282090,0.482113,0.423795,560233.000000,0.907248
510,0.294571,0.480618,0.421421,663518.000000,0.907579


TrainOutput(global_step=510, training_loss=0.03363183619929295, metrics={'train_runtime': 1508.9182, 'train_samples_per_second': 5.368, 'train_steps_per_second': 0.338, 'total_flos': 5.031497834724557e+16, 'train_loss': 0.03363183619929295, 'epoch': 6.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v2-final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v2-final")

eval_sample = test_df.sample(30, random_state=0).reset_index(drop=True)

correct = 0
valid_json = 0

for i, row in eval_sample.iterrows():
    raw_out = generate(model, tokenizer, row["resume_text"], row["job_description_text"])
    parsed = try_parse_json(raw_out)
    predicted = parsed["fit"] if parsed and "fit" in parsed else "PARSE_ERROR"
    if parsed:
        valid_json += 1
    if predicted == row["label"]:
        correct += 1
    print(f"{i+1}/30 — true: {row['label']:15s} predicted: {predicted}")

print()
print(f"JSON-validity rate: {valid_json}/30 = {valid_json/30:.1%}")
print(f"Label accuracy: {correct}/30 = {correct/30:.1%}")

1/30 — true: Good Fit        predicted: Potential Fit
2/30 — true: No Fit          predicted: Potential Fit
3/30 — true: No Fit          predicted: Good Fit
4/30 — true: No Fit          predicted: Good Fit
5/30 — true: Good Fit        predicted: Potential Fit
6/30 — true: Good Fit        predicted: No Fit
7/30 — true: Good Fit        predicted: Potential Fit
8/30 — true: Potential Fit   predicted: No Fit
9/30 — true: Good Fit        predicted: No Fit
10/30 — true: Good Fit        predicted: No Fit
11/30 — true: No Fit          predicted: Good Fit
12/30 — true: No Fit          predicted: Potential Fit
13/30 — true: Good Fit        predicted: Potential Fit
14/30 — true: No Fit          predicted: Potential Fit
15/30 — true: No Fit          predicted: Potential Fit
16/30 — true: No Fit          predicted: Potential Fit
17/30 — true: No Fit          predicted: Good Fit
18/30 — true: No Fit          predicted: Potential Fit
19/30 — true: No Fit          predicted: No Fit
20/30 — true: No Fi

In [2]:
def build_target_json(idx, label, resume, jd, verdict):
    matched, missing = matched_missing(resume, jd)
    obj = {
        "matched_skills": matched[:8],
        "missing_skills": missing[:8],
        "verdict": verdict,
        "fit": label,   # moved to LAST — model reasons through evidence before committing to the label
    }
    return json.dumps(obj, ensure_ascii=False)

SYSTEM_PROMPT = (
    "You are an AI recruiting assistant. Given a resume and a job description, respond with "
    "ONLY a JSON object with keys: matched_skills (list), missing_skills (list), verdict (one sentence), "
    "fit (\"Good Fit\", \"Potential Fit\", or \"No Fit\", based on the evidence above). No extra text."
)

training_rows = []
for _, vrow in cache_df.iterrows():
    idx = int(vrow["orig_index"])
    row = train_df.iloc[idx]
    target = build_target_json(idx, row["label"], row["resume_text"], row["job_description_text"], vrow["verdict"])
    training_rows.append({"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row["resume_text"], row["job_description_text"])},
        {"role": "assistant", "content": target},
    ]})

sft_dataset = Dataset.from_list(training_rows).train_test_split(test_size=0.1, seed=42)
print(sft_dataset)
print(training_rows[0]["messages"][2]["content"])

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 150
    })
})
{"matched_skills": [], "missing_skills": [], "verdict": "The candidate's experience in operation and maintenance of electrical systems, report analysis, and troubleshooting aligns with the job's requirements for performing routine tests, troubleshooting manufacturing problems, and executing tests according to procedures.", "fit": "Good Fit"}


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"], formatting_func=formatting_func)

trainer.train()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


Applying formatting function to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [4]:
# import os

# ckpt_root = "/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3"
# resume_path = None
# if os.path.exists(ckpt_root):
#     checkpoints = [d for d in os.listdir(ckpt_root) if d.startswith("checkpoint-")]
#     if checkpoints:
#         latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
#         resume_path = os.path.join(ckpt_root, latest)

# print("Resuming from:", resume_path if resume_path else "nowhere — starting fresh")

# trainer.train(resume_from_checkpoint=resume_path)

import os

ckpt_root = "/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3"
checkpoints = [d for d in os.listdir(ckpt_root) if d.startswith("checkpoint-")]
latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
resume_path = os.path.join(ckpt_root, latest)

print("Resuming from:", resume_path)
trainer.train(resume_from_checkpoint=resume_path)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Resuming from: /content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3/checkpoint-400


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,0.174552,0.365752,0.310744,1119402.000000,0.932679
510,0.171671,0.364929,0.310285,1223437.000000,0.932968


TrainOutput(global_step=510, training_loss=0.03998665085025862, metrics={'train_runtime': 2594.5269, 'train_samples_per_second': 3.122, 'train_steps_per_second': 0.197, 'total_flos': 5.123667279815885e+16, 'train_loss': 0.03998665085025862, 'epoch': 6.0})

In [ ]:
  print(dir())

['AutoModelForCausalLM', 'AutoTokenizer', 'BitsAndBytesConfig', 'Dataset', 'In', 'LoraConfig', 'MODEL_ID', 'Out', 'SKILL_VOCAB', 'SYSTEM_PROMPT', '_', '__', '___', '__builtin__', '__builtins__', '__doc__', '__loader__', '__name__', '__package__', '__spec__', '_dh', '_exit_code', '_i', '_i1', '_i2', '_i3', '_i4', '_i5', '_i6', '_i7', '_ih', '_ii', '_iii', '_oh', 'bnb_config', 'build_target_json', 'build_user_prompt', 'cache_df', 'checkpoints', 'ckpt_dir', 'ckpt_root', 'drive', 'exit', 'extract_skills', 'generate', 'get_ipython', 'get_peft_model', 'idx', 'json', 'latest', 'load_dataset', 'lora_config', 'matched_missing', 'model', 'name', 'os', 'param', 'pd', 'prepare_model_for_kbit_training', 'quit', 'raw', 're', 'resume_path', 'row', 'sft_dataset', 'target', 'test_df', 'tokenizer', 'torch', 'train_df', 'training_rows', 'try_parse_json', 'vrow']


In [3]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"], formatting_func=formatting_func)

print("trainer is ready")

Applying formatting function to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

trainer is ready


In [5]:
trainer.save_model("/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3-final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3-final")

eval_sample = test_df.sample(30, random_state=0).reset_index(drop=True)

correct = 0
valid_json = 0

for i, row in eval_sample.iterrows():
    raw_out = generate(model, tokenizer, row["resume_text"], row["job_description_text"])
    parsed = try_parse_json(raw_out)
    predicted = parsed["fit"] if parsed and "fit" in parsed else "PARSE_ERROR"
    if parsed:
        valid_json += 1
    if predicted == row["label"]:
        correct += 1
    print(f"{i+1}/30 — true: {row['label']:15s} predicted: {predicted}")

print()
print(f"JSON-validity rate: {valid_json}/30 = {valid_json/30:.1%}")
print(f"Label accuracy: {correct}/30 = {correct/30:.1%}")

1/30 — true: Good Fit        predicted: No Fit
2/30 — true: No Fit          predicted: Potential Fit
3/30 — true: No Fit          predicted: No Fit
4/30 — true: No Fit          predicted: No Fit
5/30 — true: Good Fit        predicted: No Fit
6/30 — true: Good Fit        predicted: No Fit
7/30 — true: Good Fit        predicted: Good Fit
8/30 — true: Potential Fit   predicted: No Fit
9/30 — true: Good Fit        predicted: Good Fit
10/30 — true: Good Fit        predicted: No Fit
11/30 — true: No Fit          predicted: No Fit
12/30 — true: No Fit          predicted: Potential Fit
13/30 — true: Good Fit        predicted: No Fit
14/30 — true: No Fit          predicted: No Fit
15/30 — true: No Fit          predicted: No Fit
16/30 — true: No Fit          predicted: Potential Fit
17/30 — true: No Fit          predicted: Good Fit
18/30 — true: No Fit          predicted: Good Fit
19/30 — true: No Fit          predicted: No Fit
20/30 — true: No Fit          predicted: No Fit
21/30 — true: Potent

In [6]:
from peft import PeftModel

# reload a clean base model, then attach the EARLIER adapter checkpoint instead of the final one
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", dtype=torch.float16)
earlier_model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3/checkpoint-300")

eval_sample = test_df.sample(30, random_state=0).reset_index(drop=True)
correct = 0
valid_json = 0

for i, row in eval_sample.iterrows():
    raw_out = generate(earlier_model, tokenizer, row["resume_text"], row["job_description_text"])
    parsed = try_parse_json(raw_out)
    predicted = parsed["fit"] if parsed and "fit" in parsed else "PARSE_ERROR"
    if parsed:
        valid_json += 1
    if predicted == row["label"]:
        correct += 1

print(f"Checkpoint-300 — JSON-validity: {valid_json}/30, Label accuracy: {correct}/30 = {correct/30:.1%}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/qwen25-1.5b-resume-fit-lora-v3/checkpoint-300'

In [ ]:
!nvidia-smi

Thu Aug 13 17:42:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----